# NB05 — Bayesian analysis of the culture-derived resistance-reduction index (ΔT)

**Public-release adaptation of the primary NB05 analysis.** Separate gastrointestinal (GI) and peritoneal (PE) Student-t models reproduce the workflow for Table 1 of the Transmid manuscript. The underlying Bayesian models, priors and sampling settings are unchanged.

**Inputs:** 32 GI observations (16 mice × two days) and 32 PE terminal observations (32 mice). The CSV files contain only de-identified analytical variables; no original study sheets, construct sequences, source codes or private Drive paths are distributed.

**Interpretation:** model locations are unconstrained and are not biological conversion fractions; later PE target and non-target observations were at different times (approximately 16–20 h versus 24 h).

**Run:** Run from the extracted repository root, or upload the ZIP into Colab session storage; Cell 2 locates and extracts the package when needed. Run cells in order. The first cells locate the repository folder and validate local files.

The notebook has not been refitted as part of packaging; new runs may differ slightly from reported posterior summaries due to Monte Carlo variation.


In [ ]:
#@title Cell 1 — Install and import dependencies { display-mode: "form" }
%pip -q install cmdstanpy arviz pandas numpy matplotlib
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from cmdstanpy import CmdStanModel, cmdstan_path, install_cmdstan
SEED = 20261123
print('NB05 setup complete.')


In [ ]:
#@title Cell 2 — Locate repository files (extract ZIP if needed) { display-mode: "form" }
from pathlib import Path
import zipfile

# Works in Colab with the ZIP uploaded to session storage,
# or directly from an already-extracted repository on a local computer.
SEARCH_ROOTS = [p for p in [Path.cwd(), Path('/content')] if p.is_dir()]
REPO_NAMES = ['Transmid_Bayesian_Resensitisation',
              'Transmid_Bayesian_Resensitisation_GitHub_Ready']

def find_project():
    candidates = []
    for root in SEARCH_ROOTS:
        candidates += [root, root.parent]
        candidates += [root / name for name in REPO_NAMES]
        candidates += [p for name in REPO_NAMES for p in root.glob('*/' + name)]
    for p in candidates:
        if ((p / 'data/NB05_GI_Delta_T.csv').is_file()
                and (p / 'data/NB05_PE_Delta_T.csv').is_file()
                and (p / 'models/NB05_GI_descriptive_deltaT.stan').is_file()
                and (p / 'models/NB05_PE_descriptive_deltaT.stan').is_file()):
            return p
    return None

# Prefer the supplied ZIP if present: it may be newer than previously
# extracted folders in a reused Colab runtime.
zip_files = []
for root in SEARCH_ROOTS:
    zip_files += list(root.glob('Transmid_Bayesian_Resensitisation*.zip'))
zip_files = sorted(set(zip_files), key=lambda p: (
    'v1.2' in p.name, p.stat().st_mtime), reverse=True)
if zip_files:
    zip_path = zip_files[0]
    with zipfile.ZipFile(zip_path) as z:
        # Ensure the ZIP contains only paths within the extraction directory.
        for member in z.infolist():
            target = (zip_path.parent / member.filename).resolve()
            if not target.is_relative_to(zip_path.parent.resolve()):
                raise ValueError('Unsafe ZIP member path: ' + member.filename)
        z.extractall(zip_path.parent)
    print('Extracted:', zip_path.name)

PROJECT = find_project()
if PROJECT is None:
    raise FileNotFoundError(
        'Repository files were not found. Upload the GitHub-ready ZIP '
        'to Colab, or run this notebook from the repository root.')

DATA_DIR, STAN_DIR, RESULT_DIR = (
    PROJECT / 'data', PROJECT / 'models', PROJECT / 'results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT)
print('Cell 2 completed successfully.')


In [ ]:
#@title Cell 3 — Load and check de-identified analysis tables { display-mode: "form" }
gid = pd.read_csv(DATA_DIR/'NB05_GI_Delta_T.csv', dtype={'animal_id':str})
ped = pd.read_csv(DATA_DIR/'NB05_PE_Delta_T.csv', dtype={'animal_id':str})
assert len(gid)==32 and len(ped)==32
assert gid.delta_T.notna().all() and ped.delta_T.notna().all()
assert set(gid.group)==set(ped.group)=={'Target','Non-target'}
assert gid.animal_id.nunique()==16 and ped.animal_id.nunique()==32
assert all(gid.groupby(['group','endpoint']).size().values==8)
assert all(ped.groupby(['group','endpoint']).size().values==8)
assert len(gid[['animal_id','actual_time']].drop_duplicates())==32
print('GI observations:',len(gid),'PE observations:',len(ped))
print('Negative ΔT: GI',int((gid.delta_T<0).sum()),'PE',int((ped.delta_T<0).sum()))
display(ped.groupby(['group','endpoint'])['actual_time'].agg(['min','max','count']))


In [ ]:
#@title Cell 4 — Construct independent Stan inputs { display-mode: "form" }
# Build group/endpoint indices and retain animal IDs for GI repeated measurements.
GROUPS = ['Target','Non-target']
ENDPOINTS = {'GI':['Day 10','Day 14'], 'PE':['Early','Later']}
CELLS = {study:[(e,g) for e in ENDPOINTS[study] for g in GROUPS] for study in ('GI','PE')}

def make_stan_data(df, study):
    cellmap = {key:i+1 for i,key in enumerate(CELLS[study])}
    grpmap = {g:i+1 for i,g in enumerate(GROUPS)}
    ans={'N':len(df),'K':4,'G':2,
         'cell':[cellmap[(r.endpoint,r.group)] for r in df.itertuples()],
         'group':[grpmap[r.group] for r in df.itertuples()],
         'y':df.delta_T.astype(float).to_list()}
    if study=='GI':
        ids=sorted(df.animal_id.unique(),key=lambda s:int(s))
        idmap={a:i+1 for i,a in enumerate(ids)}
        ans['A']=len(ids)
        ans['animal']=[idmap[a] for a in df.animal_id]
    return ans
GI_STAN_DATA = make_stan_data(gid,'GI')
PE_STAN_DATA = make_stan_data(ped,'PE')
for study,dd in (('GI',GI_STAN_DATA),('PE',PE_STAN_DATA)):
    (RESULT_DIR/f'{study}_stan_input.json').write_text(json.dumps(dd,indent=2))
    print(study,'N=',dd['N'],'cells:',CELLS[study])


In [ ]:
#@title Cell 5 — Fit two Stan models { display-mode: "form" }
# Compile and fit independent descriptive models. Retain all negative observations.
try:
    print('CmdStan:',cmdstan_path())
except ValueError:
    install_cmdstan(cores=2,progress=True)
models={
  'GI':CmdStanModel(stan_file=str(STAN_DIR/'NB05_GI_descriptive_deltaT.stan')),
  'PE':CmdStanModel(stan_file=str(STAN_DIR/'NB05_PE_descriptive_deltaT.stan'))
}
fits={}
for study,stan_data in (('GI',GI_STAN_DATA),('PE',PE_STAN_DATA)):
    outdir=RESULT_DIR/study
    outdir.mkdir(parents=True,exist_ok=True)
    fits[study]=models[study].sample(
        data=stan_data, chains=4, parallel_chains=2,
        iter_warmup=1000, iter_sampling=1000,
        seed=SEED, adapt_delta=0.97, max_treedepth=12,
        output_dir=str(outdir), show_progress=True)
    print(study,'sampling complete')


In [ ]:
#@title Cell 6 — Check convergence and summaries { display-mode: "form" }
# Inspect R-hat, effective sample sizes, sampler divergences, and posterior locations.
for study,fit in fits.items():
    print('\n===',study,'===')
    summary=fit.summary()
    display(summary.loc[[i for i in summary.index if i.startswith(('mu[','sigma[','tau'))]])
    summary.to_csv(RESULT_DIR/f'{study}_stan_summary.csv')
    diagnostics=fit.diagnose()
    print(diagnostics)
    divergence=int(np.sum(fit.method_variables()['divergent__']))
    print('Divergences:',divergence)
    if divergence>0 or (summary['R_hat'].dropna()>1.01).any():
        print('WARNING: posterior estimates require sampler troubleshooting before interpretation.')


In [ ]:
#@title Cell 7 — Summarise posterior group contrasts { display-mode: "form" }
# Estimate target-minus-non-target differences separately by endpoint.
def quantile_stats(draws,label):
    arr=np.asarray(draws,dtype=float)
    a,b,c=np.quantile(arr,[.025,.5,.975])
    return {'comparison':label,'q025':a,'median':b,'q975':c,
            'Pr_difference_gt_0':float(np.mean(arr>0))}
all_locations=[];all_contrasts=[]
for study,fit in fits.items():
    arr=fit.stan_variable('mu')
    for j,(endpoint,grp) in enumerate(CELLS[study]):
        rec=quantile_stats(arr[:,j],f'{endpoint} / {grp}')
        rec.update(study=study,endpoint=endpoint,group=grp)
        all_locations.append(rec)
    for k,endpoint in enumerate(ENDPOINTS[study]):
        diff=arr[:,2*k]-arr[:,2*k+1]
        rec=quantile_stats(diff,f'Target − non-target at {endpoint}')
        rec.update(study=study,endpoint=endpoint,
                   caution=('Not time-matched: target 16–20 h; non-target 24 h'
                            if study=='PE' and endpoint=='Later' else ''))
        all_contrasts.append(rec)
locations=pd.DataFrame(all_locations)
contrasts=pd.DataFrame(all_contrasts)
display(locations[['study','endpoint','group','q025','median','q975']])
display(contrasts[['study','comparison','q025','median','q975','Pr_difference_gt_0','caution']])
locations.to_csv(RESULT_DIR/'posterior_locations.csv',index=False)
contrasts.to_csv(RESULT_DIR/'posterior_target_contrasts.csv',index=False)


In [ ]:
#@title Cell 8 — Posterior predictive checks and export { display-mode: "form" }
# Compare replicated and observed groups and save diagnostic plots and audit tables.
pp=[]
for study,(df,fit) in {'GI':(gid,fits['GI']),'PE':(ped,fits['PE'])}.items():
    reps=fit.stan_variable('y_rep')
    stat=make_stan_data(df,study)
    y=np.asarray(stat['y']); idx=np.asarray(stat['cell'])
    for k,(endpoint,group) in enumerate(CELLS[study],start=1):
        sel=(idx==k)
        obsmean=float(y[sel].mean())
        simmean=reps[:,sel].mean(axis=1)
        q=np.quantile(simmean,[.025,.5,.975])
        pp.append({'study':study,'endpoint':endpoint,'group':group,'n':int(sel.sum()),
                   'observed_mean':obsmean,'pp_mean_q025':q[0],
                   'pp_mean_median':q[1],'pp_mean_q975':q[2],
                   'pp_exceedance':float((simmean>=obsmean).mean())})
pp_df=pd.DataFrame(pp)
display(pp_df)
pp_df.to_csv(RESULT_DIR/'posterior_predictive_summary.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=False)
for ax,study,df in zip(axes,('GI','PE'),(gid,ped)):
    sub=locations[locations.study==study].reset_index(drop=True)
    for i,r in sub.iterrows():
        ax.plot([r.q025,r.q975],[i,i],lw=2)
        ax.plot(r['median'],i,'o')
    labels=[f'{r.endpoint} / {r.group}' for r in sub.itertuples()]
    ax.set_yticks(range(len(labels)),labels)
    ax.axvline(0,linestyle=':',lw=1)
    ax.axvline(1,linestyle=':',lw=1)
    ax.set_xlabel('Observed-ΔT model location (95% CrI)')
    ax.set_title(f'{study}: independent posterior fit')
    ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULT_DIR/'NB05_posterior_locations.png',dpi=180,bbox_inches='tight')
plt.show()
QC=pd.concat([gid,ped],ignore_index=True)
QC['negative']=QC.delta_T<0
QC['above_one']=QC.delta_T>1
QC.to_csv(RESULT_DIR/'NB05_input_audit.csv',index=False)
print('Results saved to:',RESULT_DIR)
print('Reminder: group locations are not infection, conjugation or gene-inactivation rates.')
